In [1]:

import pandas as pd


In [2]:

url = "https://owid-public.owid.io/data/energy/owid-energy-data.csv"


In [5]:

import pandas as pd
import urllib.request

request = urllib.request.Request(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
)

with urllib.request.urlopen(request, timeout=30) as response:
    cols = pd.read_csv(response, nrows=0).columns.tolist()

print(cols[:30])


['country', 'year', 'iso_code', 'population', 'gdp', 'biofuel_cons_change_pct', 'biofuel_cons_change_twh', 'biofuel_cons_per_capita', 'biofuel_consumption', 'biofuel_elec_per_capita', 'biofuel_electricity', 'biofuel_share_elec', 'biofuel_share_energy', 'carbon_intensity_elec', 'coal_cons_change_pct', 'coal_cons_change_twh', 'coal_cons_per_capita', 'coal_consumption', 'coal_elec_per_capita', 'coal_electricity', 'coal_prod_change_pct', 'coal_prod_change_twh', 'coal_prod_per_capita', 'coal_production', 'coal_share_elec', 'coal_share_energy', 'electricity_demand', 'electricity_demand_per_capita', 'electricity_generation', 'electricity_share_energy']


In [6]:

wanted = ["country", "year", "iso_code", "population",
          "primary_energy_consumption", "electricity_demand",
          "renewables_consumption", "fossil_fuel_consumption"]


In [7]:

wanted = [c for c in wanted if c in cols]


In [8]:

energy = pd.read_csv(url, usecols=wanted)


HTTPError: HTTP Error 403: Forbidden

In [9]:

request = urllib.request.Request(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
)

with urllib.request.urlopen(request, timeout=60) as response:
    energy = pd.read_csv(response, usecols=wanted)

print(energy.shape)
print(energy.columns.tolist())



(23377, 8)
['country', 'year', 'iso_code', 'population', 'electricity_demand', 'fossil_fuel_consumption', 'primary_energy_consumption', 'renewables_consumption']


In [10]:

energy.shape


(23377, 8)

In [11]:

energy.info()


<class 'pandas.DataFrame'>
RangeIndex: 23377 entries, 0 to 23376
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   country                     23377 non-null  str    
 1   year                        23377 non-null  int64  
 2   iso_code                    17265 non-null  str    
 3   population                  18894 non-null  float64
 4   electricity_demand          6378 non-null   float64
 5   fossil_fuel_consumption     6405 non-null   float64
 6   primary_energy_consumption  13255 non-null  float64
 7   renewables_consumption      6405 non-null   float64
dtypes: float64(5), int64(1), str(2)
memory usage: 1.4 MB


In [12]:

energy.memory_usage(deep=True).sum()/1024**2


np.float64(3.4453792572021484)

In [13]:

energy[["country", "year"]].duplicated().sum()


np.int64(0)

In [14]:
energy.isna().mean().sort_values(ascending=False).head(20)

electricity_demand            0.727168
fossil_fuel_consumption       0.726013
renewables_consumption        0.726013
primary_energy_consumption    0.432990
iso_code                      0.261454
population                    0.191770
country                       0.000000
year                          0.000000
dtype: float64

In [15]:
country_rows = energy.loc[energy["iso_code"].notna()].copy()

In [16]:
recent = country_rows.loc[
    country_rows["year"].between(2015, 2024),
    ["country", "iso_code", "year"] + [c for c in wanted if c not in {"country", "iso_code", "year"}]
].copy()

In [17]:
recent.sort_values(["country", "year"]).head()

,country,iso_code,year,population,primary_energy_consumption,electricity_demand,renewables_consumption,fossil_fuel_consumption
141,Afghanistan,AFG,2015,33831767.0,30.953520,4.96,NaN,NaN
142,Afghanistan,AFG,2016,34700613.0,28.075348,5.54,NaN,NaN
143,Afghanistan,AFG,2017,35688941.0,36.517418,5.87,NaN,NaN
144,Afghanistan,AFG,2018,36743040.0,46.492512,6.16,NaN,NaN
145,Afghanistan,AFG,2019,37856125.0,47.111530,6.37,NaN,NaN


In [18]:
e = recent.copy()

In [19]:
missing = e.isna().mean().sort_values(ascending=False)
print(missing)

renewables_consumption        0.637448
fossil_fuel_consumption       0.637448
primary_energy_consumption    0.063332
electricity_demand            0.030748
population                    0.008720
year                          0.000000
country                       0.000000
iso_code                      0.000000
dtype: float64


In [20]:
if {"renewables_consumption", "fossil_fuel_consumption"}.issubset(e.columns):
    denom = e["renewables_consumption"] + e["fossil_fuel_consumption"]
    e["renewable_share_selected"] = e["renewables_consumption"].div(denom).where(denom.gt(0))

In [21]:
e[["country", "year", "renewables_consumption",
   "fossil_fuel_consumption", "renewable_share_selected"]].head()

,country,year,renewables_consumption,fossil_fuel_consumption,renewable_share_selected
141,Afghanistan,2015,NaN,NaN,NaN
142,Afghanistan,2016,NaN,NaN,NaN
143,Afghanistan,2017,NaN,NaN,NaN
144,Afghanistan,2018,NaN,NaN,NaN
145,Afghanistan,2019,NaN,NaN,NaN


In [22]:
latest_year = e.groupby("country")["year"].max().rename("latest_year")

In [23]:
latest = e.merge(
    latest_year,
    on="country",
    how="left",
    validate="many_to_one"
)


In [24]:
latest = latest.loc[
    latest["year"].eq(latest["latest_year"])
].copy()

In [25]:
metrics = [
    c for c in ["primary_energy_consumption", "electricity_demand"]
    if c in e.columns
]

In [26]:
period = e.groupby("country")[metrics].agg(["count", "mean", "min", "max"])

In [27]:
period.head()

primary_energy_consumption                                      \
                                    count        mean         min         max   
country                                                                         
Afghanistan                             9   41.088599   28.075348   47.569305   
Albania                                 9   25.337180   21.116934   29.290279   
Algeria                                10  685.771613  616.995850  784.532227   
American Samoa                          9    1.483694    1.376167    1.588330   
Angola                                  9   92.593099   76.321831  108.490601   

               electricity_demand                                
                            count    mean        min        max  
country                                                          
Afghanistan                    10   6.284   4.960000   7.190000  
Albania                        10   7.835   7.290000   8.670000  
Algeria                        10  81.340  68.809998  93.839996  
American Samoa                 10   0.168   0.160000   0.180000  
Angola                         10  14.284   9.780000  17.309999

In [28]:
value_cols = [
    c for c in ["primary_energy_consumption", "electricity_demand"]
    if c in e.columns
]

In [29]:
long = e.melt(
    id_vars=["country", "iso_code", "year"],
    value_vars=value_cols,
    var_name="column",
    value_name="value"
)

In [30]:
long.head()

,country,iso_code,year,column,value
0,Afghanistan,AFG,2015,primary_energy_consumption,30.953520
1,Afghanistan,AFG,2016,primary_energy_consumption,28.075348
2,Afghanistan,AFG,2017,primary_energy_consumption,36.517418
3,Afghanistan,AFG,2018,primary_energy_consumption,46.492512
4,Afghanistan,AFG,2019,primary_energy_consumption,47.111530


In [31]:
cb = codebook.drop_duplicates("column")

NameError: name 'codebook' is not defined

In [32]:
import os

print(os.listdir())

['.ipynb_checkpoints', 'adult.data', 'adult.names', 'adult.test', 'adult_cleaned.csv', 'adult_combined.csv', 'bank', 'bank-full.csv', 'bank-names.txt', 'bank.csv', 'bank_campaign_segment.csv', 'bank_marketing_cleaned.csv', 'bank_marketing_enriched.csv', 'bank_marketing_final_integrated.csv', 'bank_marketing_job_summary.csv', 'da_env', 'owid-energy-codebook.csv', 'top_20_income_groups.csv', 'Untitled.ipynb', 'Untitled1.ipynb', 'Untitled2.ipynb', 'Untitled3.ipynb', 'Untitled4.ipynb', 'work_pattern_summary.csv']


In [33]:
codebook = pd.read_csv("owid-energy-codebook.csv")

In [34]:
codebook.head()

,column,title,description,unit,source
0,country,Country,Geographic location.,NaN,Our World in Data - Regions (2025)
1,year,Year,Year of observation.,NaN,Our World in Data - Regions (2025)
2,iso_code,ISO code,ISO 3166-1 alpha-3 three-letter country codes.,NaN,International Organization for Standardization...
3,population,Population,"Population by country, available from 10,000 B...",people,Population based on various sources (2024) [ht...
4,gdp,Gross domestic product (GDP),Total economic output of a country or region p...,international-$ in 2011 prices ($),Bolt and van Zanden – Maddison Project Databas...


In [35]:
cb = codebook.drop_duplicates("column")

In [36]:
long_meta = long.merge(
    cb,
    on="column",
    how="left",
    validate="many_to_one",
    indicator=True
)

In [37]:
long_meta["_merge"].value_counts()

_merge
both          4358
left_only        0
right_only       0
Name: count, dtype: int64

In [38]:
long_meta.head()

,country,iso_code,year,column,value,title,description,unit,source,_merge
0,Afghanistan,AFG,2015,primary_energy_consumption,30.953520,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
1,Afghanistan,AFG,2016,primary_energy_consumption,28.075348,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
2,Afghanistan,AFG,2017,primary_energy_consumption,36.517418,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
3,Afghanistan,AFG,2018,primary_energy_consumption,46.492512,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
4,Afghanistan,AFG,2019,primary_energy_consumption,47.111530,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both


In [39]:
e.shape

(2179, 9)

In [40]:
e.columns.tolist()

['country',
 'iso_code',
 'year',
 'population',
 'primary_energy_consumption',
 'electricity_demand',
 'renewables_consumption',
 'fossil_fuel_consumption',
 'renewable_share_selected']

In [41]:
long_meta.shape

(4358, 10)

In [42]:
long_meta[["country", "iso_code", "year", "column"]].isna().sum()

country     0
iso_code    0
year        0
column      0
dtype: int64

In [43]:
long_meta[["country", "year", "column"]].duplicated().sum()

np.int64(0)

In [44]:
long_meta.info()

<class 'pandas.DataFrame'>
RangeIndex: 4358 entries, 0 to 4357
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   country      4358 non-null   str     
 1   iso_code     4358 non-null   str     
 2   year         4358 non-null   int64   
 3   column       4358 non-null   str     
 4   value        4153 non-null   float64 
 5   title        4358 non-null   str     
 6   description  4358 non-null   str     
 7   unit         4358 non-null   str     
 8   source       4358 non-null   str     
 9   _merge       4358 non-null   category
dtypes: category(1), float64(1), int64(1), str(7)
memory usage: 310.9 KB


In [45]:
long_meta.groupby("column")["value"].agg(
    total="size",
    available="count",
    missing=lambda s: s.isna().sum()
)

,total,available,missing
column,,,
electricity_demand,2179,2112,67
primary_energy_consumption,2179,2041,138


In [46]:
long_meta.groupby("column")["value"].apply(
    lambda s: s.isna().mean() * 100
)

column
electricity_demand            3.074805
primary_energy_consumption    6.333180
Name: value, dtype: float64

In [48]:
cleaning_log = pd.DataFrame({
    "step": [
        "Original dataset",
        "Keep rows with ISO codes",
        "Keep years 2015–2024",
        "Check missing values"
    ],
    "details": [
        f"Loaded {len(energy)} rows and {energy.shape[1]} columns",
        f"Kept {len(country_rows)} rows with non-missing ISO codes",
        f"Kept {len(recent)} rows for years 2015–2024",
        "Measured missing-value percentages; did not automatically fill missing values"
    ]
})

cleaning_log

,step,details
0,Original dataset,Loaded 23377 rows and 8 columns
1,Keep rows with ISO codes,Kept 17265 rows with non-missing ISO codes
2,Keep years 2015–2024,Kept 2179 rows for years 2015–2024
3,Check missing values,Measured missing-value percentages; did not au...


In [49]:
health_summary = pd.DataFrame({
    "check": [
        "Rows",
        "Columns",
        "Duplicate country-year keys",
        "Missing values",
        "Unique countries",
        "Year range"
    ],
    "result": [
        len(e),
        e.shape[1],
        e[["country", "year"]].duplicated().sum(),
        e.isna().sum().sum(),
        e["country"].nunique(),
        f"{e['year'].min()}–{e['year'].max()}"
    ]
})

health_summary

,check,result
0,Rows,2179
1,Columns,9
2,Duplicate country-year keys,0
3,Missing values,4391
4,Unique countries,220
5,Year range,2015–2024


In [50]:
country_summary = (
    e.groupby("country")
    .agg(
        years_observed=("year", "nunique"),
        avg_primary_energy=("primary_energy_consumption", "mean"),
        max_primary_energy=("primary_energy_consumption", "max"),
        avg_electricity_demand=("electricity_demand", "mean")
    )
    .reset_index()
)

country_summary.head()

,country,years_observed,avg_primary_energy,max_primary_energy,avg_electricity_demand
0,Afghanistan,10,41.088599,47.569305,6.284
1,Albania,10,25.337180,29.290279,7.835
2,Algeria,10,685.771613,784.532227,81.340
3,American Samoa,10,1.483694,1.588330,0.168
4,Angola,10,92.593099,108.490601,14.284


In [51]:
year_summary = (
    e.groupby("year")
    .agg(
        countries=("country", "nunique"),
        avg_primary_energy=("primary_energy_consumption", "mean"),
        avg_electricity_demand=("electricity_demand", "mean")
    )
    .reset_index()
)

year_summary

,year,countries,avg_primary_energy,avg_electricity_demand
0,2015,220,696.412361,112.769908
1,2016,220,705.011367,115.958732
2,2017,220,720.741841,119.383099
3,2018,220,739.096273,124.203003
4,2019,220,747.792869,125.892675
5,2020,220,720.999196,125.432064
6,2021,220,757.271971,132.667041
7,2022,220,771.278085,135.703521
8,2023,220,785.674706,139.371651
9,2024,199,2157.466429,157.242041


In [52]:
e.loc[
    e["year"].eq(2024),
    ["country", "primary_energy_consumption", "electricity_demand"]
].sort_values(
    "primary_energy_consumption",
    ascending=False
).head(10)

,country,primary_energy_consumption,electricity_demand
4606,China,48987.101562,10069.879883
21997,United States,26528.611328,4404.830078
9698,India,11336.056641,2034.219971
18158,Russia,9049.149414,1192.910034
10623,Japan,4794.816406,1016.390015
3117,Brazil,3919.053955,757.309998
3939,Canada,3886.195068,624.359985
19450,South Korea,3616.218262,625.380005
9949,Iran,3589.785889,393.390015
18641,Saudi Arabia,3290.423828,454.589996


In [53]:
e.loc[e["year"].eq(2024), [
    "primary_energy_consumption",
    "electricity_demand"
]].agg(["count", "mean", "min", "max"])

,primary_energy_consumption,electricity_demand
count,79.000000,196.000000
mean,2157.466429,157.242041
min,29.569088,0.010000
max,48987.101562,10069.879883


In [54]:
e["primary_energy_per_person"] = (
    e["primary_energy_consumption"]
    .div(e["population"])
    .where(e["population"].gt(0))
)

e[[
    "country",
    "year",
    "population",
    "primary_energy_consumption",
    "primary_energy_per_person"
]].head()

,country,year,population,primary_energy_consumption,primary_energy_per_person
141,Afghanistan,2015,33831767.0,30.953520,9.149247e-07
142,Afghanistan,2016,34700613.0,28.075348,8.090735e-07
143,Afghanistan,2017,35688941.0,36.517418,1.023214e-06
144,Afghanistan,2018,36743040.0,46.492512,1.265342e-06
145,Afghanistan,2019,37856125.0,47.111530,1.244489e-06


In [55]:
print("Rows in analysis dataset:", len(e))
print(
    "Missing primary_energy_per_person:",
    e["primary_energy_per_person"].isna().sum()
)
print(
    "Negative primary_energy_per_person values:",
    e["primary_energy_per_person"].lt(0).sum()
)
print(
    "Non-missing primary_energy_per_person values:",
    e["primary_energy_per_person"].count()
)

Rows in analysis dataset: 2179
Missing primary_energy_per_person: 156
Negative primary_energy_per_person values: 0
Non-missing primary_energy_per_person values: 2023


In [56]:
print(
    "Missing renewable_share_selected:",
    e["renewable_share_selected"].isna().sum()
)

print(
    "Negative renewable shares:",
    e["renewable_share_selected"].lt(0).sum()
)

print(
    "Renewable shares above 1:",
    e["renewable_share_selected"].gt(1).sum()
)

print(
    "Non-missing renewable shares:",
    e["renewable_share_selected"].count()
)

Missing renewable_share_selected: 1389
Negative renewable shares: 0
Renewable shares above 1: 0
Non-missing renewable shares: 790


In [57]:
final_energy = e.merge(
    year_summary,
    on="year",
    how="left",
    validate="many_to_one",
    suffixes=("", "_year")
)

print("Final dataset shape:", final_energy.shape)
print("Duplicate country-year keys:",
      final_energy[["country", "year"]].duplicated().sum())

final_energy.head()

Final dataset shape: (2179, 13)
Duplicate country-year keys: 0


,country,iso_code,year,population,primary_energy_consumption,electricity_demand,renewables_consumption,fossil_fuel_consumption,renewable_share_selected,primary_energy_per_person,countries,avg_primary_energy,avg_electricity_demand
0,Afghanistan,AFG,2015,33831767.0,30.953520,4.96,NaN,NaN,NaN,9.149247e-07,220,696.412361,112.769908
1,Afghanistan,AFG,2016,34700613.0,28.075348,5.54,NaN,NaN,NaN,8.090735e-07,220,705.011367,115.958732
2,Afghanistan,AFG,2017,35688941.0,36.517418,5.87,NaN,NaN,NaN,1.023214e-06,220,720.741841,119.383099
3,Afghanistan,AFG,2018,36743040.0,46.492512,6.16,NaN,NaN,NaN,1.265342e-06,220,739.096273,124.203003
4,Afghanistan,AFG,2019,37856125.0,47.111530,6.37,NaN,NaN,NaN,1.244489e-06,220,747.792869,125.892675


In [58]:
print("Original analysis rows:", len(e))
print("Final dataset rows:", len(final_energy))
print(
    "Duplicate country-year keys:",
    final_energy[["country", "year"]].duplicated().sum()
)

print(
    "Missing yearly averages:",
    final_energy[
        ["avg_primary_energy", "avg_electricity_demand"]
    ].isna().sum().sum()
)

print(
    "Required derived features present:",
    {
        "renewable_share_selected",
        "primary_energy_per_person"
    }.issubset(final_energy.columns)
)

assert len(final_energy) == len(e)
assert final_energy[["country", "year"]].duplicated().sum() == 0
assert {
    "renewable_share_selected",
    "primary_energy_per_person"
}.issubset(final_energy.columns)

print("All structural validation checks passed.")

Original analysis rows: 2179
Final dataset rows: 2179
Duplicate country-year keys: 0
Missing yearly averages: 0
Required derived features present: True
All structural validation checks passed.


In [59]:
final_energy.to_csv("owid_energy_final_integrated.csv", index=False)

print("Final integrated dataset exported successfully.")

Final integrated dataset exported successfully.


In [60]:
import os
import pandas as pd

file_path = "owid_energy_final_integrated.csv"

print("File exists:", os.path.exists(file_path))

exported_energy = pd.read_csv(file_path)

print("Exported rows:", exported_energy.shape[0])
print("Exported columns:", exported_energy.shape[1])
print("Column names:", exported_energy.columns.tolist())

assert os.path.exists(file_path)
assert exported_energy.shape == final_energy.shape

print("PASS: Export validation successful.")

File exists: True
Exported rows: 2179
Exported columns: 13
Column names: ['country', 'iso_code', 'year', 'population', 'primary_energy_consumption', 'electricity_demand', 'renewables_consumption', 'fossil_fuel_consumption', 'renewable_share_selected', 'primary_energy_per_person', 'countries', 'avg_primary_energy', 'avg_electricity_demand']
PASS: Export validation successful.


In [61]:
cleaning_log.to_csv("owid_energy_cleaning_log.csv", index=False)
health_summary.to_csv("owid_energy_health_summary.csv", index=False)
country_summary.to_csv("owid_energy_country_summary.csv", index=False)
year_summary.to_csv("owid_energy_year_summary.csv", index=False)

print("All four supporting CSV files exported successfully.")

All four supporting CSV files exported successfully.


In [62]:
import os
import pandas as pd

files = [
    "owid_energy_final_integrated.csv",
    "owid_energy_cleaning_log.csv",
    "owid_energy_health_summary.csv",
    "owid_energy_country_summary.csv",
    "owid_energy_year_summary.csv"
]

for file in files:
    if os.path.exists(file):
        check = pd.read_csv(file)
        print(f"PASS: {file} | Rows: {len(check)} | Columns: {len(check.columns)}")
    else:
        print(f"FAIL: {file} not found")

PASS: owid_energy_final_integrated.csv | Rows: 2179 | Columns: 13
PASS: owid_energy_cleaning_log.csv | Rows: 4 | Columns: 2
PASS: owid_energy_health_summary.csv | Rows: 6 | Columns: 2
PASS: owid_energy_country_summary.csv | Rows: 220 | Columns: 5
PASS: owid_energy_year_summary.csv | Rows: 10 | Columns: 4
